# Notebook 01 — Data Exploration

Mount Drive, install deps, download Oxford-IIIT Pet, verify trimap + seg-map quality.

In [ ]:
# ── Colab bootstrap ───────────────────────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !git clone https://github.com/reddy-nithin/stable-diffusion.git /content/stable-diffusion
    %cd /content/stable-diffusion
    !pip install -q -r requirements.txt

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path().resolve()))

# Download dataset
!python scripts/download_data.py

In [ ]:
from src.data.dataset import OxfordPetDataset
from src.data.taxonomy import load_taxonomy, all_breeds, all_conditions

ds = OxfordPetDataset(root='data')
tax = load_taxonomy()

print(f'Dataset size : {len(ds)}')
print(f'Unique breeds: {len(ds.classes)}')
print(f'Breeds       : {ds.classes}')
print()
print('Conditions:', list(all_conditions(tax).keys()))

In [ ]:
# ── Breed distribution ────────────────────────────────────────────────────────
import collections, matplotlib.pyplot as plt

counts = collections.Counter(ds.breed_names)
breeds_sorted = sorted(counts.items(), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar([b for b, _ in breeds_sorted], [c for _, c in breeds_sorted])
ax.set_xticklabels([b for b, _ in breeds_sorted], rotation=90, fontsize=7)
ax.set_ylabel('Images')
ax.set_title('Oxford-IIIT Pet — breed distribution')
plt.tight_layout()
plt.savefig('outputs/breed_distribution.png', dpi=80)
plt.show()

In [ ]:
# ── Visualise 20 samples: image | trimap | seg-map ────────────────────────────
import random, numpy as np
from src.data.masks import trimap_to_seg_map

pathlib.Path('outputs').mkdir(exist_ok=True)

random.seed(42)
indices = random.sample(range(len(ds)), 20)

fig, axes = plt.subplots(20, 3, figsize=(9, 60))
col_titles = ['Image', 'Trimap', 'ADE20K seg map']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=9, fontweight='bold')

for row, idx in enumerate(indices):
    image, trimap, breed, species = ds[idx]
    seg_map = trimap_to_seg_map(trimap)

    axes[row, 0].imshow(image)
    axes[row, 0].set_ylabel(f'{breed}\n({species})', fontsize=6, rotation=0,
                             labelpad=60, va='center')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(np.array(trimap))
    axes[row, 1].axis('off')

    axes[row, 2].imshow(np.array(seg_map))
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig('outputs/data_exploration_grid.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved → outputs/data_exploration_grid.png')

In [ ]:
# ── Trimap pixel-value sanity check ──────────────────────────────────────────
from collections import Counter

for idx in indices[:5]:
    _, trimap, breed, _ = ds[idx]
    vals = Counter(np.array(trimap).flatten().tolist())
    total = sum(vals.values())
    fg_pct = vals.get(1, 0) / total * 100
    print(f'{breed:30s}  fg={fg_pct:.1f}%  vals={dict(vals)}')